# Gold Layer - Business Intelligence Marts

**Purpose**: Pre-aggregated, dashboard-ready business metrics and analytics optimized for BI tool consumption.

**Target Audience**: Business analysts, executives, dashboard consumers

**Layer Position**: Final consumption layer in the medallion architecture (Bronze → Silver → Semantic → Warehouse → **Gold**)

---

## 📊 Overview

The Gold layer contains **denormalized, pre-computed fact tables** designed for:
* **Fast dashboard queries** - Pre-aggregated metrics eliminate expensive joins
* **Business-friendly structure** - Organized by business domain (salary, skills, hiring)
* **Time-series analysis** - Optimized for trending and temporal comparisons
* **Self-service analytics** - Simple schemas that business users can query directly

### Architecture Principles

* **Denormalized for Performance** - Dimensional attributes embedded in facts
* **Pre-Aggregated** - Rollups computed at ingestion time, not query time
* **Temporal Optimization** - Window functions and lag metrics for trending
* **Multiple Grain Levels** - Drill-down capability (total → sector → role → location)
* **Metadata Timestamps** - All tables include `created_at` and `batch_id` for lineage

---

## 📁 Gold Notebooks

### **Core Analytics Marts**

#### 1. `gold_salary_trends`
**Purpose**: Salary trend analytics with percentile distributions  
**Target Table**: `workspace.gold.gold_salary_trends`  
**Grain**: Date × Sector × Role × Location × Company  
**Key Metrics**: Median salaries (min/max), P25/P75/P90 percentiles, observation counts, MoM/QoQ changes  
**Rollups**: Sector-level, role-level, location-level aggregations  
**Use Cases**: Compensation benchmarking, salary trend dashboards, competitive intelligence

#### 2. `gold_skill_demand`
**Purpose**: Skill demand trends and co-occurrence patterns  
**Target Table**: `workspace.gold.gold_skill_demand`  
**Grain**: Date × Skill × Sector × Role  
**Key Metrics**: Job postings requiring skill, co-occurrence rankings, skill trend velocity  
**Use Cases**: Skills gap analysis, curriculum planning, upskilling priorities

#### 3. `gold_hiring_trends`
**Purpose**: Job posting velocity and hiring activity indicators  
**Target Table**: `workspace.gold.gold_hiring_trends`  
**Grain**: Date × Sector × Location  
**Key Metrics**: New postings, closures, open positions, posting velocity, avg time-to-fill  
**Use Cases**: Labor market health tracking, hiring demand forecasting

#### 4. `gold_location_trends`
**Purpose**: Geographic hiring patterns and remote work analysis  
**Target Table**: `workspace.gold.gold_location_trends`  
**Grain**: Date × Location × Work Mode (Remote/Hybrid/Onsite)  
**Key Metrics**: Posting counts by location, remote work %, geographic concentration  
**Use Cases**: Regional labor market reports, remote work adoption tracking

#### 5. `gold_sector_overview`
**Purpose**: Sector-level summary metrics and cross-sector comparisons  
**Target Table**: `workspace.gold.gold_sector_overview`  
**Grain**: Date × Sector  
**Key Metrics**: Total postings, avg salary, top skills, hiring velocity, sector growth  
**Use Cases**: Industry benchmarking, sector health reports, executive dashboards

#### 6. `gold_company_hiring`
**Purpose**: Company-level hiring activity and employer brand insights  
**Target Table**: `workspace.gold.gold_company_hiring`  
**Grain**: Date × Company × Sector  
**Key Metrics**: Active postings, new postings, hiring velocity, role distribution  
**Use Cases**: Employer intelligence, competitive hiring analysis

---

### **Specialized Analytics**

#### 7. `gold_hospitality_hiring`
**Purpose**: Hospitality sector-specific hiring trends  
**Target Table**: `workspace.gold.gold_hospitality_hiring`  
**Grain**: Date × Hospitality Subsector × Location  
**Key Metrics**: Postings by hotel/restaurant/travel segments, seasonal patterns, recovery indicators  
**Use Cases**: Hospitality industry reports, tourism recovery tracking

#### 8. `gold_hospitality_skills`
**Purpose**: Hospitality-specific skill demand patterns  
**Target Table**: `workspace.gold.gold_hospitality_skills`  
**Grain**: Date × Skill × Hospitality Subsector  
**Key Metrics**: In-demand hospitality skills, certification requirements, skill gaps  
**Use Cases**: Hospitality workforce planning, training program design

#### 9. `gold_hospitality_companies`
**Purpose**: Major hospitality employer hiring activity  
**Target Table**: `workspace.gold.gold_hospitality_companies`  
**Grain**: Date × Company  
**Key Metrics**: Postings by major brands (Marriott, Hilton, etc.), expansion indicators  
**Use Cases**: Employer-specific intelligence, industry consolidation analysis

---

### **Operational Monitoring**

#### 10. `gold_pipeline_health`
**Purpose**: Data pipeline quality and operational health metrics  
**Target Table**: `workspace.gold.gold_pipeline_health`  
**Grain**: Date × Pipeline × Layer  
**Key Metrics**: Run success rate, data freshness, quality scores, processing latency  
**Use Cases**: DataOps monitoring, SLA tracking, data quality dashboards

---

## 🔧 Usage Patterns

### **Query Pattern: Time-Series Analysis**

```sql
-- Example: Salary trends over time for Tech sector
SELECT 
  DATE(CAST(salary_date_sk AS STRING), 'yyyyMMdd') AS trend_date,
  salary_max_median,
  delta_vs_prev_month,
  pct_change_vs_prev_month
FROM workspace.gold.gold_salary_trends gst
JOIN workspace.warehouse.dim_sector s ON gst.sector_sk = s.sector_sk
WHERE s.sector_name = 'Technology'
  AND gst.role_sk IS NULL  -- Sector-level rollup
  AND gst.salary_date_sk >= 20230101
ORDER BY trend_date DESC;
```

### **Query Pattern: Drill-Down Analysis**

```sql
-- Example: Sector → Role → Location drill-down
-- Level 1: Sector totals
SELECT sector_sk, SUM(observation_count) AS total_obs
FROM workspace.gold.gold_salary_trends
WHERE role_sk IS NULL AND location_sk IS NULL
GROUP BY sector_sk;

-- Level 2: Drill into specific roles within sector
SELECT role_sk, SUM(observation_count) AS total_obs
FROM workspace.gold.gold_salary_trends
WHERE sector_sk = 5 AND location_sk IS NULL
GROUP BY role_sk;
```

### **Refresh Strategy**

Gold tables are typically refreshed:
* **Daily** - Trend tables (salary, hiring, location)
* **Weekly** - Aggregated summaries (sector overview, company hiring)
* **Ad-hoc** - Specialized analytics (hospitality marts)

Each table includes:
* `created_at` - Last refresh timestamp
* `batch_id` - UUID linking to orchestration run

---

## 🔄 Data Flow & Dependencies

### **Upstream Dependencies**

```
Bronze (API Snapshots)
  ↓
Silver (Deduped Jobs)
  ↓
Semantic (Enrichments: Skills, Roles, Companies)
  ↓
Warehouse (Star Schema: Dims + Facts)
  ↓
Gold (Pre-Aggregated Marts) ← YOU ARE HERE
```

### **Primary Source Tables**

* `workspace.warehouse.fact_job_postings` - Core job posting facts
* `workspace.warehouse.fact_salary` - Salary observations
* `workspace.warehouse.bridge_job_skill` - Job-skill relationships
* `workspace.warehouse.dim_sector` - Sector dimension
* `workspace.warehouse.dim_role` - Role dimension
* `workspace.warehouse.dim_location` - Location dimension
* `workspace.warehouse.dim_company` - Company dimension
* `workspace.warehouse.dim_date` - Date dimension

### **Table Dependencies**

| Gold Table | Primary Warehouse Source |
|------------|-------------------------|
| `gold_salary_trends` | `fact_salary` |
| `gold_skill_demand` | `bridge_job_skill` |
| `gold_hiring_trends` | `fact_job_postings` |
| `gold_location_trends` | `fact_job_postings` + `dim_location` |
| `gold_sector_overview` | `fact_job_postings` + `dim_sector` |
| `gold_company_hiring` | `fact_job_postings` + `dim_company` |
| `gold_hospitality_*` | Filtered views of core facts (sector filter) |
| `gold_pipeline_health` | `warehouse.fact_pipeline_runs` |

---

## 📐 Schema Conventions

### **Standard Columns**

All Gold tables include:

| Column | Type | Purpose |
|--------|------|--------|
| `*_date_sk` | INT | Date surrogate key (yyyyMMdd format) |
| `*_sk` | BIGINT | Foreign keys to warehouse dimensions |
| `observation_count` | BIGINT | Number of raw records aggregated |
| `created_at` | TIMESTAMP | Table refresh timestamp |
| `batch_id` | STRING | UUID linking to orchestration run |

### **Temporal Columns**

Trend tables include:
* `prev_month_*` - Prior month comparison value
* `prev_quarter_*` - Prior quarter comparison value
* `delta_vs_prev_month` - Absolute change from previous month
* `pct_change_vs_prev_month` - Percentage change from previous month

### **Rollup Indicators**

NULL dimension keys indicate rollup levels:
* `sector_sk = NULL, role_sk = NULL` → Total across all sectors and roles
* `sector_sk = 5, role_sk = NULL` → Sector 5 total (all roles)
* `sector_sk = -1` → Placeholder for "all sectors" in role-level rollups

### **Naming Conventions**

* **Table Names**: `gold_<domain>_<aggregation>` (e.g., `gold_salary_trends`)
* **Measures**: Descriptive names (e.g., `salary_max_median`, `posting_velocity`)
* **Dimensions**: Foreign key suffix `_sk` (e.g., `sector_sk`, `role_sk`)

---

## ✅ Best Practices

### **When Creating New Gold Tables**

1. **Start with Business Question** - Design aggregations to answer specific BI questions
2. **Pre-Compute Expensive Operations** - Window functions, percentiles, complex joins
3. **Include Multiple Grain Levels** - Enable drill-down without reprocessing
4. **Add Temporal Metrics** - Period-over-period comparisons for trending
5. **Document Minimum Sample Sizes** - Filter for statistical validity (e.g., `observation_count >= 5`)
6. **Test Query Performance** - Gold tables should return in < 2 seconds for dashboards

### **Optimization Techniques**

* **Partition by Date** - Most queries filter on date ranges
* **Z-Order on Common Filters** - Sector, location, company keys
* **Compact Delta Tables** - Regularly OPTIMIZE to merge small files
* **Use DECIMAL for Currency** - Avoid floating-point precision issues
* **Materialize Slowly** - Don't recompute historical data on every run

### **Quality Checks**

* **Row Count Validation** - Compare to warehouse layer (Gold should be <= Warehouse)
* **Null Checks** - Key metrics should never be NULL (use 0 or filter)
* **Temporal Continuity** - Verify no missing dates in time-series
* **Rollup Consistency** - Sector totals should match sum of role-level data

---

## 🔧 Maintenance & Operations

### **Routine Maintenance**

```sql
-- Optimize Gold tables (run weekly)
OPTIMIZE workspace.gold.gold_salary_trends ZORDER BY (sector_sk, role_sk);
OPTIMIZE workspace.gold.gold_skill_demand ZORDER BY (skill_sk, sector_sk);

-- Vacuum old versions (run monthly, retain 30 days)
VACUUM workspace.gold.gold_salary_trends RETAIN 720 HOURS;
```

### **Monitoring Queries**

```sql
-- Check data freshness
SELECT 
  'gold_salary_trends' AS table_name,
  MAX(created_at) AS last_refresh,
  DATEDIFF(HOUR, MAX(created_at), CURRENT_TIMESTAMP()) AS hours_since_refresh
FROM workspace.gold.gold_salary_trends
UNION ALL
SELECT 
  'gold_hiring_trends',
  MAX(created_at),
  DATEDIFF(HOUR, MAX(created_at), CURRENT_TIMESTAMP())
FROM workspace.gold.gold_hiring_trends;

-- Check row counts and growth
SELECT 
  'gold_salary_trends' AS table_name,
  COUNT(*) AS row_count,
  COUNT(DISTINCT batch_id) AS unique_batches,
  MIN(salary_date_sk) AS earliest_date,
  MAX(salary_date_sk) AS latest_date
FROM workspace.gold.gold_salary_trends;
```

### **Troubleshooting**

**Issue**: Dashboard queries slow  
**Solution**: Check for missing Z-ordering, run OPTIMIZE

**Issue**: Missing dates in time-series  
**Solution**: Verify warehouse layer has continuous dates, check upstream pipeline failures

**Issue**: Rollup totals don't match detail rows  
**Solution**: Review NULL key conventions, verify aggregation logic

---

## 📚 Related Documentation

* **Warehouse Layer** - See `notebooks/warehouse/README_WAREHOUSE.md` for upstream star schema documentation
* **Semantic Layer** - See `notebooks/semantic/README_SEMANTIC.md` for enrichment logic
* **Data Dictionary** - See `docs/data_dictionary.md` for column definitions
* **Dashboard Guides** - See `dashboards/README.md` for BI tool integration

---

## 🏁 Quick Start

1. **Explore a Gold Table**:
   ```sql
   SELECT * FROM workspace.gold.gold_salary_trends LIMIT 100;
   ```

2. **Run a Notebook**:
   ```python
   dbutils.notebook.run("./gold_salary_trends", timeout_seconds=3600)
   ```

3. **Check Freshness**:
   ```sql
   SELECT MAX(created_at) FROM workspace.gold.gold_salary_trends;
   ```

---

**Last Updated**: 2026-06-05  
**Maintained By**: Data Platform Team  
**Questions?**: Contact #data-platform on Slack